# Pilot: Perception & Reasoning Entropy Signal Check (FERMAT + Qwen2.5-VL)

Purpose: the smallest, fastest end-to-end run to check whether perception entropy
(instability in transcribing handwritten math) and reasoning entropy (instability in
grading an answer for errors) are visibly higher on items the model gets wrong. This
is a scoped-down pilot, not the full study — no bootstrap CIs, no baseline suite, no
conformal calibration, no human double grading.

**This notebook does not carry its own copy of the pipeline logic.** Every non-GPU
piece (data loading/filtering, prompt formatting, output parsing, entropy calculation)
lives in the `pilot/` package and is unit-tested locally. This notebook clones that
same code repo fresh each session and installs it, so there is a single source of
truth — editing a module locally and re-running this notebook next session picks up
the change automatically, with no manual copy/paste re-sync step.

Only the GPU-dependent parts live here: installing GPU deps, loading the model, and
running the sampling loop.

In [13]:
# Install cell: GPU-dependent packages only.
# `datasets` is intentionally also in the local requirements.txt -- each
# environment installs its own copy independently, no conflict.
# torch is not installed explicitly: Colab GPU runtimes ship with it preinstalled.
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils

In [14]:
# Auth & code/results access cell.
import os
from getpass import getpass

from huggingface_hub import login

# --- Hugging Face auth ---
# login() is required here because ai4bharat/FERMAT is a gated dataset.
#
# HOW TO USE THIS CELL: run it, then an input box appears -- in VS Code it
# shows at the TOP of the window, not inline under the cell. Paste your
# token INTO THAT BOX and press Enter. The string below is only the prompt
# label; do NOT put your token there. getpass keeps the token in memory
# only, so it never lands in this file or in git history.
#
# google.colab.userdata.get() (the usual Colab-secrets approach) is not used
# here because it only works in the native Colab browser UI, not over VS
# Code's Jupyter connection.
HF_TOKEN = getpass("Paste your Hugging Face token here, then press Enter: ").strip()

# Fail loudly on an empty or malformed token. Without this, huggingface_hub
# sends an "Authorization: Bearer " header with nothing after it, and the
# failure surfaces as a cryptic `LocalProtocolError: Illegal header value
# b'Bearer '` from deep inside httpx -- which gives no hint that the real
# problem is simply that no token was entered at the prompt.
if not HF_TOKEN:
    raise ValueError(
        "No token entered. Re-run this cell and paste your token into the "
        "input box (top of the window in VS Code), then press Enter."
    )
if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "That does not look like a Hugging Face token (they start with 'hf_'). "
        "Make sure you pasted the token itself into the input box."
    )

login(token=HF_TOKEN)

# --- Drive mount: used ONLY to cache model weights across sessions ---
from google.colab import drive

drive.mount("/content/drive")
DRIVE_MODEL_CACHE = "/content/drive/MyDrive/uncertainty-math-vlm/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

# --- Clone the repo (code + results live in the same repo for this pilot) ---
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
REPO_IS_PRIVATE = False  # set True if the repo is private
if REPO_IS_PRIVATE:
    GH_TOKEN = getpass("Paste your GitHub token here, then press Enter: ").strip()
    clone_url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@")
else:
    clone_url = REPO_URL

# Remove any stale clone from a previous (possibly failed) run so this cell
# is safe to re-run -- git clone silently no-ops into a pre-existing
# directory, which would otherwise leave `repo/` incomplete without error.
!rm -rf repo
!git clone {clone_url} repo

# %pip (not !pip) installs into the *running kernel's* environment -- !pip
# can silently target a different Python install.
%pip install -q -e repo/

# An editable install writes an `__editable__.pilot-*.pth` file into
# site-packages, but .pth files are only processed by the `site` module at
# INTERPRETER STARTUP. The kernel is already running, so it never sees them
# and `import pilot` fails with ModuleNotFoundError even though the install
# reported success. Putting the repo on sys.path directly makes the package
# importable right now, with no kernel restart needed.
import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'repo'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 45 (delta 18), reused 37 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 21.72 KiB | 21.72 MiB/s, done.
Resolving deltas: 100% (18/18), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot package imported from: /content/repo/pilot


In [15]:
# Model load cell.
# Start with the 3B model for the first smoke test -- same prompt format and
# code path as the 7B, but noticeably faster to load and run, so early bugs
# get caught cheaply. Swap MODEL_ID to the 7B line below once the pipeline
# runs cleanly end to end on the 3B.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # swap in once the 3B pipeline is clean

# If memory is tight on the 7B, load in 4-bit instead:
# from transformers import BitsAndBytesConfig
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     quantization_config=quantization_config,
#     device_map="auto",
#     cache_dir=DRIVE_MODEL_CACHE,
# )

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=DRIVE_MODEL_CACHE,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
# Sampling loop cell.
#
# K=5 samples at temperature=0.7 for each of the transcription and grading
# prompts, plus 2 samples at temperature=0 (greedy, back-to-back) as a
# low-temperature sanity anchor -- 2 draws, not 1, because a single-sample
# entropy is mathematically zero by definition and could never catch a real
# bug (e.g. an accidental sampling flag, batching nondeterminism). With 2
# draws it's a genuine, if noisy, instability check.
#
# Each generation call is wrapped in a bounded retry scoped strictly to
# infrastructure-level failures (dropped connection, transient OOM, Colab
# runtime hiccup) -- never around parsing. A response that fails to parse is
# real data about that sample's behavior under that prompt, not a transient
# failure to retry past; retrying until a clean parse appears would silently
# bias every entropy estimate downward.
import time
import gc

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

# This loop makes 14 model calls per item (2 prompts x (5 sampled + 2 greedy)),
# so N=25 is ~350 calls and can take a while on a T4. Consider setting N=2 for
# a first end-to-end check that the whole pipeline runs, then raising it.
N = 25  # smoke-test size; bump to 75-100 for the Step 3 7B run
SEED = 42
K = 5
TEMP = 0.7
N_TEMP0 = 2
MAX_RETRIES = 3
RETRY_PAUSE_SECONDS = 5

INFRA_EXCEPTIONS = (
    ConnectionError,
    TimeoutError,
    torch.cuda.OutOfMemoryError,
    OSError,
)


def generate(messages, do_sample: bool, temperature: float | None):
    """Run one generation call, retrying only on infrastructure-level failures."""
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            text_prompt = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text_prompt],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to(model.device)

            gen_kwargs = {"max_new_tokens": 512, "do_sample": do_sample}
            if do_sample:
                gen_kwargs["temperature"] = temperature

            with torch.no_grad():
                output_ids = model.generate(**inputs, **gen_kwargs)

            trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
            return processor.batch_decode(
                trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )[0]
        except INFRA_EXCEPTIONS as exc:
            last_exc = exc
            gc.collect()
            torch.cuda.empty_cache()
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_PAUSE_SECONDS)
    raise last_exc


def run_batch(messages, n, do_sample, temperature, stage, item_idx, n_items, pbar):
    """Draw n samples for one prompt, ticking the progress bar after each call."""
    outputs = []
    for j in range(n):
        pbar.set_postfix_str(f"item {item_idx + 1}/{n_items} | {stage} {j + 1}/{n}")
        outputs.append(generate(messages, do_sample=do_sample, temperature=temperature))
        pbar.update(1)
    return outputs


sample = pilot.data.load_fermat_sample(n=N, seed=SEED)
n_items = len(sample)

# The bar counts individual generation calls rather than items: an item-level
# bar would only advance once every 14 calls, which on a slow run looks frozen
# for minutes at a time and gives a poor ETA.
calls_per_item = 2 * (K + N_TEMP0)
total_calls = n_items * calls_per_item
print(f"{n_items} items x {calls_per_item} calls = {total_calls} generation calls")

raw_results = []
with tqdm(total=total_calls, desc="generating", unit="call") as pbar:
    for item_idx, item in enumerate(sample):
        image = item["image"]
        transcription_messages = pilot.prompts.build_transcription_messages(image)
        grading_messages = pilot.prompts.build_grading_messages(image)

        transcription_samples_raw = run_batch(
            transcription_messages, K, True, TEMP,
            "transcribe T=0.7", item_idx, n_items, pbar,
        )
        grading_samples_raw = run_batch(
            grading_messages, K, True, TEMP,
            "grade T=0.7", item_idx, n_items, pbar,
        )
        transcription_temp0_raw = run_batch(
            transcription_messages, N_TEMP0, False, None,
            "transcribe T=0", item_idx, n_items, pbar,
        )
        grading_temp0_raw = run_batch(
            grading_messages, N_TEMP0, False, None,
            "grade T=0", item_idx, n_items, pbar,
        )

        raw_results.append(
            {
                "item": item,
                "transcription_samples_raw": transcription_samples_raw,
                "grading_samples_raw": grading_samples_raw,
                "transcription_temp0_raw": transcription_temp0_raw,
                "grading_temp0_raw": grading_temp0_raw,
            }
        )

print(f"Collected raw samples for {len(raw_results)} items.")

### Scoring cell — note on the temperature-0 anchor

`temp0_entropy_transcription` / `temp0_entropy_grading` below are computed over
**2** greedy draws per item, not 1. With only 1 sample, `cluster_entropy` would be
mathematically zero by definition regardless of anything the model actually did —
a tautology, not a finding. With 2 draws, a nonzero value is a real (if noisy)
signal of low-temperature instability — e.g. an accidentally-enabled sampling flag,
or batching nondeterminism — and should be read as a **pipeline smoke test**, not
as an empirical claim about the model's true low-temperature behavior. If these are
not at or near zero for nearly every item, something in the sampling or parsing is
broken and needs fixing before trusting anything else.

Per-item parse-failure counts (`n_transcription_parse_failures`,
`n_grading_parse_failures`, out of K=5) are also recorded here so Step 4 can check
the **aggregate** parse-failure rate across the whole run — a systematic regex or
prompt-format bug can make every sample in an item fail to parse, which collapses
to a single confident `<PARSE_FAILURE>` cluster (entropy 0) and would otherwise
hide inside numbers that look clean.

In [ ]:
# Scoring cell.
import pilot.parsing
import pilot.entropy

scored_results = []
for entry in raw_results:
    item = entry["item"]

    transcription_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_samples_raw"]
    ]
    grading_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]
    transcription_temp0_parsed = [
        pilot.parsing.parse_transcription(t) for t in entry["transcription_temp0_raw"]
    ]
    grading_temp0_parsed = [pilot.parsing.parse_grading(t) for t in entry["grading_temp0_raw"]]

    perception_entropy = pilot.entropy.cluster_entropy(transcription_parsed)
    reasoning_entropy = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_parsed]
    )
    temp0_entropy_transcription = pilot.entropy.cluster_entropy(transcription_temp0_parsed)
    temp0_entropy_grading = pilot.entropy.cluster_entropy(
        [None if d is None else str(d) for d in grading_temp0_parsed]
    )

    majority_transcription, _ = pilot.entropy.majority_cluster(transcription_parsed)
    transcription_correct = majority_transcription == pilot.entropy.normalize_string(
        item["pert_a"]
    )

    majority_grading, _ = pilot.entropy.majority_cluster(
        [None if d is None else str(d) for d in grading_parsed]
    )
    grading_correct = majority_grading == pilot.entropy.normalize_string(str(item["has_error"]))

    n_transcription_parse_failures = sum(1 for t in transcription_parsed if t is None)
    n_grading_parse_failures = sum(1 for d in grading_parsed if d is None)

    scored_results.append(
        {
            "orig_q": item["orig_q"],
            "pert_a": item["pert_a"],
            "has_error": item["has_error"],
            "handwriting_style": item["handwriting_style"],
            "image_quality": item["image_quality"],
            "perception_entropy": perception_entropy,
            "reasoning_entropy": reasoning_entropy,
            "temp0_entropy_transcription": temp0_entropy_transcription,
            "temp0_entropy_grading": temp0_entropy_grading,
            "transcription_correct": transcription_correct,
            "grading_correct": grading_correct,
            "n_transcription_parse_failures": n_transcription_parse_failures,
            "n_grading_parse_failures": n_grading_parse_failures,
            "all_transcription_samples_raw": entry["transcription_samples_raw"],
            "all_grading_samples_raw": entry["grading_samples_raw"],
            "temp0_transcription_raw": entry["transcription_temp0_raw"],
            "temp0_grading_raw": entry["grading_temp0_raw"],
            "model_id": MODEL_ID,
            "n_items": N,
        }
    )

print(f"Scored {len(scored_results)} items.")

In [ ]:
# Save cell: write CSV into the cloned repo's results/ dir, commit, push.
import subprocess
from datetime import datetime, timezone

import pandas as pd

df = pd.DataFrame(scored_results)

model_slug = MODEL_ID.split("/")[-1].lower().replace(".", "")
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = f"results_{model_slug}_{timestamp}.csv"

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"

df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(df)} rows)")

subprocess.run(["git", "-C", "repo", "add", f"results/{csv_name}"], check=True)
subprocess.run(
    ["git", "-C", "repo", "commit", "-m", f"Add pilot results: {csv_name}"],
    check=True,
)
subprocess.run(["git", "-C", "repo", "push"], check=True)
print("Pushed results to the repo.")